# Lab 2 — Filtering, binary images and features

**Deep Learning for Computer Vision — Day 1** · ECAM, 5MAU · 2026–2027

**Goals.**
1. Implement a 2D convolution yourself, and understand what `cv2.filter2D` computes. Tomorrow, CNNs will *learn* these kernels.
2. Denoise with linear (Gaussian) and non-linear (median) filters; compute gradients and edges (Sobel, Canny).
3. Binarise (Otsu), clean the result with morphology and count objects (connected components).
4. Understand Harris corners and find an object with ORB keypoints, even rotated and scaled.

**Priority:** sections 1, 2 and 4. Sections 3, 5 and 6 can be finished at home if needed.

> *This lab is largely adapted from* Q. Lurkin, *Image Processing – Lab 2: Filtering*, ECAM, https://quentin.lurkin.xyz/courses/imageprocessing/lab2/ (convolution exercise, Nemo, coins, Harris; images `nemo.jpg`, `coins.jpg`, `chessboard.jpg`). Other references: R. Szeliski, *Computer Vision: Algorithms and Applications*, 2nd ed., ch. 3 & 7; OpenCV tutorials, https://docs.opencv.org/

In [ ]:
# Setup check: are all the packages of requirements.txt installed in this kernel?
import importlib
import importlib.util

importlib.invalidate_caches()
REQUIRED = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib", "cv2": "opencv-contrib-python",
            "ipywidgets": "ipywidgets", "ipympl": "ipympl", "serial": "pyserial"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if "opencv-contrib-python" not in missing:
    import cv2
    if not hasattr(cv2, "aruco"):  # opencv-python (without contrib) shadows opencv-contrib-python
        missing.append("opencv-contrib-python (run `%pip uninstall -y opencv-python opencv-contrib-python` "
                       "then `%pip install \"opencv-contrib-python<5\"`)")
assert not missing, (f"Missing packages: {missing}. Run `%pip install -r ../requirements.txt` "
                     "in a cell, then restart the kernel (Kernel > Restart) and run the notebook again.")
print("Setup OK: all required packages are installed.")

In [ ]:
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

DATA = Path("data") if Path("data").exists() else Path("../data")
plt.rcParams["figure.figsize"] = (10, 5)


def show(*images, titles=None, cmap="gray", size=4):
    """Display several images side by side (BGR colour images are converted)."""
    fig, ax = plt.subplots(1, len(images), figsize=(size * len(images), size), squeeze=False)
    for k, im in enumerate(images):
        a = ax[0, k]
        a.imshow(im[:, :, ::-1] if im.ndim == 3 else im, cmap=cmap)
        a.axis("off")
        if titles:
            a.set_title(titles[k])
    plt.tight_layout()


table = cv2.imread(str(DATA / "table" / "table_normal.png"))
gray = cv2.cvtColor(table, cv2.COLOR_BGR2GRAY)
show(table, gray, titles=["table", "gray"])

## 1. 2D convolution

Filtering an image $f$ with a kernel $h$ of size $(2k+1)\times(2k+1)$ replaces each pixel by a weighted sum of its neighbours:

$$ g(y, x) = \sum_{i=-k}^{k}\sum_{j=-k}^{k} h(i, j)\, f(y + i,\, x + j) $$

Strictly speaking this is a **cross-correlation**; the true convolution flips the kernel ($f(y-i, x-j)$). For symmetric kernels both are identical. `cv2.filter2D` — and the "convolution" layers of PyTorch — actually compute the cross-correlation.

At the borders the neighbours do not exist: we must **pad** the image (zeros, replicate the edge pixels, reflect...).

### Exercise 1.1 — Naive implementation
Implement `correlate2d_naive(f, h)` with two loops over the pixels (output of the same size as `f`, borders padded by replication with `np.pad(..., mode="edge")`). Test it on a small image only: it is slow!

In [ ]:
def correlate2d_naive(f, h):
    f = f.astype(np.float32)
    k = h.shape[0] // 2
    padded = np.pad(f, k, mode="edge")
    out = np.zeros_like(f)
    # TODO
    ...
    return out


small = gray[200:300, 400:520]
box = np.ones((3, 3), np.float32) / 9
t0 = time.perf_counter()
res = correlate2d_naive(small, box)
print(f"naive: {1000 * (time.perf_counter() - t0):.1f} ms for {small.size} pixels")
ref = cv2.filter2D(small.astype(np.float32), -1, box, borderType=cv2.BORDER_REPLICATE)
assert np.allclose(res, ref, atol=1e-3)

### Exercise 1.2 — Vectorised implementation
Loop over the **kernel** entries instead of the pixels: for each $(i, j)$, add $h(i,j)$ times a shifted copy of the padded image. Only $(2k+1)^2$ iterations remain.

In [ ]:
def correlate2d(f, h):
    f = f.astype(np.float32)
    k = h.shape[0] // 2
    H, W = f.shape
    padded = np.pad(f, k, mode="edge")
    out = np.zeros_like(f)
    # TODO
    ...
    return out


t0 = time.perf_counter()
res = correlate2d(gray, box)
print(f"vectorised: {1000 * (time.perf_counter() - t0):.1f} ms for {gray.size} pixels")
t0 = time.perf_counter()
ref = cv2.filter2D(gray.astype(np.float32), -1, box, borderType=cv2.BORDER_REPLICATE)
print(f"cv2.filter2D: {1000 * (time.perf_counter() - t0):.1f} ms")
assert np.allclose(res, ref, atol=1e-3)

### Exercise 1.3 — A zoo of kernels
Apply the kernels below and explain what each one does. Note how the output range changes: some results are negative or above 255, so **filter in `float32`** and only clip/convert for display.

In [ ]:
kernels = {
    "box 5x5": np.ones((5, 5), np.float32) / 25,
    "sharpen": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32),
    "laplacian": np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32),
    "sobel x": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32),
    "sobel y": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], np.float32),
}
crop = gray[150:450, 300:800]
results = [correlate2d(crop, h) for h in kernels.values()]
for name, r in zip(kernels, results):
    print(f"{name:10s} min {r.min():7.1f}  max {r.max():7.1f}")
show(crop, *[np.abs(r) for r in results], titles=["input"] + list(kernels), size=3)

### Exercise 1.4 — Design your own kernel: a 4x4 diagonal Sobel
`sobel x` detects vertical edges and `sobel y` horizontal ones. Write **yourself** a **4x4** kernel `diag` that detects **diagonal edges** oriented like "\\" (top-left to bottom-right), with the intensity increasing towards the **top-right**, and that gives **no response** to the other diagonal "/".

Hints: look at how the weights of `sobel x` are arranged (zero on the "edge line", opposite signs on each side). Your kernel should
- sum to zero (no response on a flat region),
- be zero on its main diagonal (the "\\" line through the kernel),
- be **antisymmetric** with respect to that diagonal: `diag.T == -diag`.

The cell tests your kernel on two synthetic edges, then applies it to the table. Since our `correlate2d` assumes an odd kernel with a centre pixel, we use `cv2.filter2D` here. Question: why are kernels almost always of **odd** size?

In [ ]:
# TODO
diag = np.zeros((4, 4), np.float32)  # replace by your kernel
print(diag)

# Two synthetic edges: "\" (bright at the top-right) and "/"
yy, xx = np.mgrid[0:64, 0:64]
edge_backslash = np.where(xx > yy, 255, 0).astype(np.float32)
edge_slash = np.where(xx + yy > 64, 255, 0).astype(np.float32)
r_back = cv2.filter2D(edge_backslash, -1, diag, borderType=cv2.BORDER_REPLICATE)
r_slash = cv2.filter2D(edge_slash, -1, diag, borderType=cv2.BORDER_REPLICATE)
r_slash[:4], r_slash[-4:], r_slash[:, :4], r_slash[:, -4:] = 0, 0, 0, 0  # ignore the image borders
print(f"max response on '\\' edge: {r_back.max():.0f}   max |response| on '/' edge: {np.abs(r_slash).max():.0f}")
show(edge_backslash, r_back, edge_slash, r_slash,
     titles=["'\\' edge", "diag response", "'/' edge", "diag response"], size=3)

assert diag.shape == (4, 4)
assert abs(diag.sum()) < 1e-6, "the kernel must sum to zero"
assert r_back.max() > 0 and r_back.max() > 10 * np.abs(r_slash).max() + 1, "strong on '\\', none on '/'"

r_diag = cv2.filter2D(crop.astype(np.float32), -1, diag, borderType=cv2.BORDER_REPLICATE)
show(crop, np.abs(results[3]), np.abs(results[4]), np.abs(r_diag),
     titles=["input", "sobel x", "sobel y", "diagonal (4x4)"], size=3)

## 2. Denoising: linear vs non-linear filters

In [ ]:
rng = np.random.default_rng(0)
noisy_gauss = np.clip(crop + rng.normal(0, 20, crop.shape), 0, 255).astype(np.uint8)
noisy_sp = crop.copy()
mask = rng.random(crop.shape)
noisy_sp[mask < 0.03] = 0
noisy_sp[mask > 0.97] = 255
show(noisy_gauss, noisy_sp, titles=["Gaussian noise", "salt & pepper noise"])

### Exercise 2.1
Denoise both noisy images with a **Gaussian blur** (σ = 1, 2, 4) and a **median filter** (size 3 and 5). The loop and the display are given: you only have to write the two filter calls.
- `cv2.GaussianBlur(img, (0, 0), sigma)`: with a kernel size of `(0, 0)`, OpenCV chooses the size from σ;
- `cv2.medianBlur(img, ksize)`: `ksize` is odd (3, 5, ...).

Look at the results (zoom on the dark disc): which filter is best for which noise? What happens to the **edges** when σ grows?

In [ ]:
noisy_images = {"Gaussian noise": noisy_gauss, "salt & pepper": noisy_sp}
all_filtered = []
fig, ax = plt.subplots(2, 6, figsize=(20, 5.5))
for row, (noise_name, noisy) in enumerate(noisy_images.items()):
    filtered = {"no filter": noisy}
    for s in [1, 2, 4]:
        # TODO
        out = ...  # Gaussian blur of `noisy` with standard deviation s
        filtered[f"Gaussian σ={s}"] = out
    for k in [3, 5]:
        # TODO
        out = ...  # median filter of `noisy` with a k x k window
        filtered[f"median {k}x{k}"] = out
    all_filtered.append(filtered)
    for a, (filter_name, im) in zip(ax[row], filtered.items()):
        a.imshow(im, cmap="gray", vmin=0, vmax=255)
        a.set_title(f"{noise_name}\n{filter_name}")
        a.axis("off")
plt.tight_layout()

# Zoom on the dark disc to see the details
y0, x0 = 40, 130
fig, ax = plt.subplots(2, 6, figsize=(20, 7))
for row, filtered in enumerate(all_filtered):
    for a, (filter_name, im) in zip(ax[row], filtered.items()):
        a.imshow(im[y0:y0 + 100, x0:x0 + 140], cmap="gray", vmin=0, vmax=255)
        a.set_title(f"zoom: {filter_name}")
        a.axis("off")
plt.tight_layout()

## 3. Gradients and edges

The Sobel kernels approximate $\partial f/\partial x$ and $\partial f/\partial y$. The **gradient magnitude** $\sqrt{g_x^2 + g_y^2}$ is large on edges and its **orientation** $\mathrm{atan2}(g_y, g_x)$ is perpendicular to the edge.

### Exercise 3.1
1. Compute $g_x, g_y$ with `cv2.Sobel` (output `cv2.CV_32F`), the magnitude and the orientation.
2. `cv2.Canny(gray, low, high)` is run below with two pairs of thresholds: one very low (17, 30), one very high (350, 360). Describe both results.
3. Find thresholds `low_t, high_t` that give **all the edges of the objects** on the table (including the light square and the ArUco markers) with almost no noise pixels. What are the two thresholds of Canny for?

In [ ]:
# TODO
...
show(mag, ori, titles=["magnitude", "orientation"], size=5)

edges_low = cv2.Canny(gray, 17, 30)
edges_high = cv2.Canny(gray, 350, 360)
show(edges_low, edges_high, titles=["Canny 17/30", "Canny 350/360"], size=6)

In [ ]:
# TODO
low_t, high_t = ..., ...  # your thresholds
edges = cv2.Canny(gray, low_t, high_t)
show(edges, titles=[f"Canny {low_t}/{high_t}"], size=10)

n_edge = (edges > 0).sum()
n_square = (edges[340:440, 350:460] > 0).sum()  # around the light square, low contrast
print(f"edge pixels: {n_edge} | on the light square: {n_square}")
assert n_square > 100, "the light square is missing: the thresholds are too high"
assert n_edge < 8000, "too many edge pixels: the noise is detected, the thresholds are too low"

## 4. Binary images: thresholding, morphology, connected components

A thresholded mask is rarely perfect: isolated white specks (noise), small holes inside the objects, thin cracks... **Morphological operators** clean it. They slide a small **structuring element** (e.g. a 3x3 square) over the binary image:
- **erosion** (`cv2.erode`): a pixel stays white only if *all* the pixels under the structuring element are white → objects **shrink**, small objects disappear;
- **dilation** (`cv2.dilate`): a pixel becomes white if *at least one* pixel under the structuring element is white → objects **grow**, small holes are filled;
- **opening** = erosion, then dilation; **closing** = dilation, then erosion (`cv2.morphologyEx(img, cv2.MORPH_OPEN / cv2.MORPH_CLOSE, se)`).

### Exercise 4.1 — The four basic operators on a toy image
The toy image below (the same as in the slides) contains a square with a one-pixel hole, a rectangle with a thin crack, a thin line and an isolated speck.
1. Create a 3x3 square structuring element with `cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))`.
2. Compute the erosion and the dilation of `toy`.
3. Compute the opening and the closing with `cv2.morphologyEx`.
4. Which operator removes the speck and the line? Which one fills the hole and the crack? Do the square and the rectangle keep their size after an opening or a closing?
5. Try a 5x5 structuring element: what changes?

In [ ]:
toy = np.zeros((13, 20), np.uint8)
toy[1:8, 1:8] = 1; toy[4, 4] = 0           # square with a one-pixel hole
toy[1:8, 11:18] = 1; toy[1:4, 14] = 0      # rectangle with a thin crack
toy[11, 17] = 1                             # isolated speck (noise)
toy[11, 1:8] = 1                            # thin line
toy = np.pad(toy, 2)                        # empty margin: the shapes never touch the border


def show_grid(*images, titles):
    """Display small binary images with the pixel grid."""
    fig, ax = plt.subplots(1, len(images), figsize=(3.6 * len(images), 2.8))
    for a, im, t in zip(ax, images, titles):
        a.imshow(im, cmap="gray_r", vmin=0, vmax=1)
        a.set_xticks(np.arange(-0.5, im.shape[1]), minor=True)
        a.set_yticks(np.arange(-0.5, im.shape[0]), minor=True)
        a.grid(which="minor", color="lightgray", lw=0.5)
        a.tick_params(which="both", length=0, labelbottom=False, labelleft=False)
        a.set_title(t)
    plt.tight_layout()


# TODO
se = ...
# TODO
eroded, dilated = ..., ...
# TODO
opened, closed = ..., ...
print("white pixels: input", toy.sum(), "| eroded", eroded.sum(), "| dilated", dilated.sum(),
      "| opened", opened.sum(), "| closed", closed.sum())
show_grid(toy, eroded, dilated, opened, closed, titles=["input", "erosion", "dilation", "opening", "closing"])

assert eroded.sum() < toy.sum() < dilated.sum()
assert opened[13].sum() == 0, "the opening removes the speck and the line"
assert closed[6, 6] == 1 and closed[3:6, 16].all(), "the closing fills the hole and the crack"
assert np.array_equal(opened, cv2.dilate(cv2.erode(toy, se), se)), "opening = erosion, then dilation"

### Exercise 4.2 — Nemo
1. Build the red/orange mask of `nemo.jpg` in HSV (Lab 1).
2. Count its connected components with `cv2.connectedComponents`.
3. Clean the mask with a morphological **opening** (removes small specks) then a **closing** (fills small holes) and count again.

In [ ]:
nemo = cv2.imread(str(DATA / "nemo.jpg"))
hsv = cv2.cvtColor(nemo, cv2.COLOR_BGR2HSV)
mask = cv2.bitwise_or(cv2.inRange(hsv, (0, 120, 80), (18, 255, 255)),
                      cv2.inRange(hsv, (170, 120, 80), (179, 255, 255)))
n_raw, _ = cv2.connectedComponents(mask)
# TODO
clean = ...
n_clean, labels = cv2.connectedComponents(clean)
print("regions before:", n_raw - 1, "| after:", n_clean - 1)  # label 0 is the background
show(mask, clean, labels, titles=["raw mask", "opened + closed", "labels"], cmap="nipy_spectral")

### Exercise 4.3 — How much money?
The image contains only 1 € coins (Ø 23.25 mm) and 10 c coins (Ø 19.75 mm).
1. Try a global **Otsu** threshold on the grey image. Why does it fail?
2. Otsu works much better on the **saturation** channel (S of HSV): the coins are golden (saturated), the paper is white–grey (unsaturated). The second cell computes this mask for you. Why does it work, even with the shadow?
3. The third cell cleans the mask (closing, then opening), labels it with `cv2.connectedComponentsWithStats` and gives you the **area** (in pixels) of every coin. Loop over these areas, decide for each coin whether it is a 1 € or a 10 c coin from its size, and compute the total amount. Hint: the ratio of the areas of the two coins is $(23.25 / 19.75)^2 \approx 1.39$.

In [ ]:
coins = cv2.imread(str(DATA / "coins.jpg"))
g = cv2.cvtColor(coins, cv2.COLOR_BGR2GRAY)
t, otsu_gray = cv2.threshold(cv2.GaussianBlur(g, (7, 7), 0), 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
print("Otsu threshold on gray:", t)
show(coins, otsu_gray, titles=["coins", "Otsu on gray"])

In [ ]:
def saturation_mask(img_bgr):
    """Otsu threshold on the (blurred) saturation channel: white = coins."""
    sat = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)[:, :, 1]
    sat = cv2.GaussianBlur(sat, (7, 7), 0)
    t, m = cv2.threshold(sat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    print("Otsu threshold on saturation:", t)
    return sat, m


sat, otsu_sat = saturation_mask(coins)
show(sat, otsu_sat, titles=["saturation", "Otsu on saturation"])

In [ ]:
clean = cv2.morphologyEx(otsu_sat, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15)))
clean = cv2.morphologyEx(clean, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
n, labels, stats, centroids = cv2.connectedComponentsWithStats(clean)
areas = stats[1:, cv2.CC_STAT_AREA]  # label 0 is the background
print("areas of the coins (pixels):", areas.tolist())
show(clean, labels, titles=["closing + opening", "labels"], cmap="nipy_spectral")

In [ ]:
amount = 0.0
# TODO
for a in areas: ...
print(f"amount: {amount:.2f} EUR")
assert abs(amount - 2.40) < 1e-6

## 5. Points of interest: Harris corners

Corners are points where the gradient varies in **two** directions: they are easy to locate precisely and to match between images. The calibration this afternoon relies on detecting the corners of a checkerboard.

### Exercise 5.1 — What does Harris look at? (no code to write)
Harris computes the two derivatives $I_x, I_y$ (Sobel) and, around every pixel, sums three products over a small window: $\sum I_x^2$, $\sum I_y^2$ and $\sum I_x I_y$. The cell below displays these three maps on a zoom of `chessboard.jpg`, then the Harris response

$$ R = \textstyle\sum I_x^2 \cdot \sum I_y^2 - \left(\sum I_x I_y\right)^2 - k\left(\sum I_x^2 + \sum I_y^2\right)^2, \qquad k = 0.04 $$

and the pixels where $R$ is above 1 % of its maximum.
1. Where is $\sum I_x^2$ large? And $\sum I_y^2$? What happens at a corner of a square?
2. $\sum I_x I_y$ is large (red or blue) on the tilted edges, and close to 0 at the corners. Why does $R$ *subtract* $(\sum I_x I_y)^2$? Think of a diagonal edge: $I_x$ and $I_y$ are both large there, but it is not a corner.
3. Compare with `cv2.cornerHarris` on the whole board (last image).

In [ ]:
board = cv2.imread(str(DATA / "chessboard.jpg"))
bg = np.float32(cv2.cvtColor(board, cv2.COLOR_BGR2GRAY)) / 255

Ix = cv2.Sobel(bg, cv2.CV_32F, 1, 0, ksize=3)
Iy = cv2.Sobel(bg, cv2.CV_32F, 0, 1, ksize=3)
window = lambda a: cv2.GaussianBlur(a, (0, 0), 2)  # weighted sum over a small window around each pixel
Sxx, Syy, Sxy = window(Ix * Ix), window(Iy * Iy), window(Ix * Iy)
R = Sxx * Syy - Sxy ** 2 - 0.04 * (Sxx + Syy) ** 2

z = (slice(180, 380), slice(350, 650))  # zoom on a few squares
fig, ax = plt.subplots(2, 3, figsize=(15, 7))
ax[0, 0].imshow(board[z][:, :, ::-1]); ax[0, 0].set_title("zoom")
ax[0, 1].imshow(Sxx[z], cmap="magma"); ax[0, 1].set_title(r"$\sum I_x^2$")
ax[0, 2].imshow(Syy[z], cmap="magma"); ax[0, 2].set_title(r"$\sum I_y^2$")
v = np.abs(Sxy[z]).max()
ax[1, 0].imshow(Sxy[z], cmap="RdBu_r", vmin=-v, vmax=v); ax[1, 0].set_title(r"$\sum I_x I_y$ (red > 0, blue < 0)")
ax[1, 1].imshow(np.maximum(R[z], 0), cmap="magma"); ax[1, 1].set_title("Harris response R")
zoom_vis = board[z].copy()
zoom_vis[cv2.dilate(R[z], np.ones((5, 5))) > 0.01 * R.max()] = (0, 0, 255)
ax[1, 2].imshow(zoom_vis[:, :, ::-1]); ax[1, 2].set_title("R > 1 % of max")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()

# The same with OpenCV on the whole board
R_cv = cv2.cornerHarris(bg, 2, 3, 0.04)
vis = board.copy()
vis[cv2.dilate(R_cv, np.ones((5, 5))) > 0.01 * R_cv.max()] = (0, 0, 255)  # dilate: only for visibility
show(vis, titles=["cv2.cornerHarris"], size=7)

## 6. Find a reference object with ORB features

ORB detects keypoints (corner-like points, at several scales) and computes for each a binary **descriptor** that is normalised by the orientation of the keypoint. Descriptors are matched with the Hamming distance (number of different bits).

### Exercise 6.1 (no code to write)
The reference object is the ArUco marker at the top-left of the table. To check that ORB is robust to geometric changes, the template is **enlarged ×2 and rotated by 90°** before matching it against the table.
1. Where do the 20 best matches land on the table?
2. Why can't a simple comparison of the pixel values (template matching) find this object?
3. Try other transformations (rotation of 180°, scale ×0.7, `cv2.flip`): which ones does ORB survive?

In [ ]:
crop = table[60:170, 270:390]  # ArUco marker 0 as "reference object"
template = cv2.rotate(cv2.resize(crop, None, fx=2, fy=2), cv2.ROTATE_90_CLOCKWISE)
orb = cv2.ORB_create(1000)
k1, d1 = orb.detectAndCompute(template, None)
k2, d2 = orb.detectAndCompute(table, None)
matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = sorted(matcher.match(d1, d2), key=lambda m: m.distance)[:20]
pts = np.array([k2[m.trainIdx].pt for m in matches])
on_marker = ((pts[:, 0] > 270) & (pts[:, 0] < 390) & (pts[:, 1] > 60) & (pts[:, 1] < 170)).sum()
print(f"{len(k1)} keypoints in the template, {len(k2)} in the table; {on_marker}/20 best matches on the marker")
vis = cv2.drawMatches(template, k1, table, k2, matches, None)
show(vis, size=12)